# Telegram Topic Modeling
<!-- structured-notebook -->
## Notebook Summary
Purpose: fit a BERTopic model on collected Telegram messages and inspect the resulting topics.

Main steps:
- Preprocess messages (URL / emoji removal, lemmatization, English filter).
- Embed with `all-mpnet-base-v2` for cross-platform comparability with Reddit.
- UMAP `n_neighbors=20, n_components=10, cosine`; HDBSCAN `min_cluster_size=80, cluster_selection_method="eom"`.
- Vectorizer `ngram_range=(1,2), min_df=2, max_df=0.8, max_features=60000`.
- Representation model: `KeyBERTInspired` for post-hoc keyword refinement.

### Why `eom` rather than `leaf`

The Reddit production model uses `leaf` for finer-grained clusters. On the smaller Telegram corpus, `leaf` produced too many singleton clusters; `eom` gave more stable, interpretable topics.


# Inputs / Outputs
<!-- io-table -->

| Role | File | Upstream / downstream |
|---|---|---|
| Input | `<DATA_ROOT>/Telegram/output/messages.json` | Produced by `01_data_collection/01_channel_crawl.ipynb` |
| Output | `<DATA_ROOT>/Telegram/output/bertopic/` | Consumed by `04_topic_matching/01_cross_platform_matching.ipynb` |


## 1. Setup & Imports

In [ ]:
import sys
from pathlib import Path

# Walk up from the notebook's directory to the repo root (the first ancestor that contains `src/`) and add it to sys.path
_repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from src.shared_reddit_telegram.config import PROJECT_ROOT, TELEGRAM_PIPELINE, TELEGRAM_DATA

In [ ]:
import json
import pickle

import pandas as pd
import umap
import hdbscan
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

from src.shared_reddit_telegram.text_cleaning import clean_text, is_english, preprocess_pipeline

## 2. Load Messages

Messages are stored as a JSON array of objects, each with `source`, `msg_id`, `date`, `text`, and `forwarded_from` fields. These were collected via the hybrid Telethon pipeline (see `02_channel_crawl.ipynb`).

In [ ]:
messages_path = TELEGRAM_PIPELINE / "hybrid" / "messages.json"

with open(messages_path) as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(f"Loaded {len(df)} messages from {df['source'].nunique()} channels")
df.head()

## 3. Preprocess Text

The preprocessing pipeline is shared with the Reddit analysis track for consistency across social-media sources. It performs:

1. **URL removal** — strips all hyperlinks
2. **Emoji removal** — removes unicode emoji characters
3. **English filter** — discards non-English messages (Telegram channels often mix languages)
4. **Lemmatization** — reduces words to base forms via spaCy for better topic coherence
5. **Deduplication** — maps duplicate texts to a single representative to avoid topic skew

In [ ]:
texts = df["text"].fillna("").astype(str).tolist()

preprocessed, kept_idx, unique, map_o2u = preprocess_pipeline(texts)

print(f"Original messages:      {len(texts)}")
print(f"After English filter:   {len(kept_idx)}")
print(f"Unique after dedup:     {len(unique)}")
print(f"Ready for BERTopic:     {len(preprocessed)}")

## 4. BERTopic Configuration

Parameter choices for the Telegram corpus:

- **UMAP**: `n_neighbors=20, n_components=10, metric='cosine'` — 20 neighbors provides a good balance between local and global structure for a medium-sized corpus; 10 components retain enough variance for downstream clustering.
- **HDBSCAN**: `min_cluster_size=80, cluster_selection_method='eom'` — EOM (Excess of Mass) tends to produce more coherent clusters. The minimum cluster size of 80 prevents overly fragmented topics while still surfacing niche longevity discussions.
- **Vectorizer**: bigrams enabled (`ngram_range=(1,2)`), English stop words removed, with document-frequency bounds to filter noise.
- **Representation**: `KeyBERTInspired` produces more interpretable topic labels by leveraging embedding similarity rather than raw c-TF-IDF terms.

In [ ]:
umap_model = umap.UMAP(
    n_neighbors=20,
    n_components=10,
    metric="cosine",
    random_state=42,
    low_memory=True,
)

hdbscan_model = hdbscan.HDBSCAN(
    min_cluster_size=80,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)

vectorizer = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.8,
    max_features=60000,
)

representation_model = KeyBERTInspired()

topic_model = BERTopic(
    embedding_model="all-mpnet-base-v2",
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    representation_model=representation_model,
    nr_topics="auto",
    top_n_words=10,
    calculate_probabilities=True,
    verbose=True,
)

## 5. Fit BERTopic

In [ ]:
topics, probs = topic_model.fit_transform(preprocessed)

## 6. Save Outputs

Persist the fitted model, topic assignments, embeddings, and preprocessed documents for downstream analysis and visualization.

In [ ]:
output_dir = TELEGRAM_PIPELINE / "hybrid" / "bertopic_output"
output_dir.mkdir(parents=True, exist_ok=True)

# Save the BERTopic model
topic_model.save(
    str(output_dir / "telegram_topic_model"),
    serialization="safetensors",
    save_ctfidf=True,
    save_embedding_model="all-mpnet-base-v2",
)

# Save topic assignments as CSV
topics_df = pd.DataFrame({"doc_index": range(len(preprocessed)), "topic": topics})
topics_df.to_csv(output_dir / "topics.csv", index=False)

# Save preprocessed docs and index mappings
with open(output_dir / "preprocessed_docs.pkl", "wb") as f:
    pickle.dump(
        {
            "preprocessed": preprocessed,
            "kept_idx": kept_idx,
            "unique": unique,
            "map_o2u": map_o2u,
        },
        f,
    )

print(f"Outputs saved to {output_dir}")

## 7. Inspect Results

In [ ]:
topic_info = topic_model.get_topic_info()
print(f"Discovered {len(topic_info) - 1} topics (excluding outlier topic -1)")
print(f"Outlier documents: {(pd.Series(topics) == -1).sum()} / {len(topics)}")
print()
topic_info.head(20)

In [ ]:
# Visualize topic hierarchy
topic_model.visualize_hierarchy()

In [ ]:
# Visualize intertopic distance map
topic_model.visualize_topics()

---
<!-- nav-footer -->

← [01 data exploration](../../../../SocialMedia/Telegram/notebooks/02_exploration/01_data_exploration.ipynb) | [Project Overview](../../../../PROJECT_OVERVIEW.ipynb)
